In [1]:
# If any library is missing, run this cell once:
%pip install pandas numpy matplotlib seaborn ipywidgets

%matplotlib inline
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", category=FutureWarning)
print("Libraries imported successfully.")


Note: you may need to restart the kernel to use updated packages.
Libraries imported successfully.


In [2]:
# Optional: create a small sample dataset to test the notebook.
# You can instead load your own CSV file using the menu below.
sample_data = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=8, freq="MS"),
    "Product": ["Book", "Pen", "Notebook", "Book", "Pen", "Notebook", "Book", "Pen"],
    "Category": ["Stationery", "Stationery", "Stationery", "Stationery",
                 "Stationery", "Stationery", "Stationery", "Stationery"],
    "Units_Sold": [12, 25, 18, 15, 30, 22, 20, 28],
    "Revenue": [2400, 500, 900, 3000, 600, 1100, 4000, 560]
})
sample_data.to_csv("sample_sales_data.csv", index=False)
print("Created sample_sales_data.csv")
display(sample_data.head())


Created sample_sales_data.csv


,Date,Product,Category,Units_Sold,Revenue
0,2026-01-01,Book,Stationery,12,2400
1,2026-02-01,Pen,Stationery,25,500
2,2026-03-01,Notebook,Stationery,18,900
3,2026-04-01,Book,Stationery,15,3000
4,2026-05-01,Pen,Stationery,30,600


In [3]:
# Jupyter button callbacks cannot use terminal input().
# Enter all answers needed by an option in the Answers box, one per line.
# This notebook routes class-method input prompts through the answer queue below.
_notebook_input_answers = []

class NotebookInputCancelled(Exception):
    """Stop an option cleanly when the Answers box has no more responses."""
    pass

def notebook_input(prompt=""):
    if not _notebook_input_answers:
        raise NotebookInputCancelled(
            "This option needs more answers. Type the requested answers in the Answers box, "
            "one per line, then click Run Selected Option again. For submenu choices, include "
            "the final Back to main menu choice (for example, 8 for Explore Data or 7 for Clean Data)."
        )
    answer = _notebook_input_answers.pop(0)
    print(f"{prompt}{answer}")
    return answer

# The class methods below use this notebook-friendly replacement for input().
input = notebook_input

class SalesDataAnalyzer:
    """Encapsulates sales-data loading, cleaning, analysis and visualization."""

    def __init__(self, file_path=None):
        self.data = pd.DataFrame()
        self.file_path = None
        self.last_plot = None
        if file_path:
            self.load_data(file_path)

    def __del__(self):
        # Destructor included to demonstrate the OOP concept.
        pass

    def load_data(self, file_path):
        """Load data from a CSV file and report errors clearly."""
        try:
            if not os.path.isfile(file_path):
                print(f"File not found: {file_path}")
                return False
            loaded = pd.read_csv(file_path)
            if loaded.empty:
                print("The CSV file is empty.")
                return False
            self.data = loaded
            self.file_path = file_path
            print(f"Dataset loaded successfully! Rows: {len(self.data)}, Columns: {len(self.data.columns)}")
            print("Columns:", ", ".join(map(str, self.data.columns)))
            return True
        except (pd.errors.ParserError, UnicodeDecodeError, OSError, ValueError) as exc:
            print(f"Could not load the CSV file: {exc}")
            return False

    def _ready(self):
        if self.data.empty:
            print("Please load a dataset first (main menu option 1).")
            return False
        return True

    def explore_data(self):
        if not self._ready():
            return
        while True:
            print("\n== Explore Data ==")
            print("1. Display the first 5 rows")
            print("2. Display the last 5 rows")
            print("3. Display column names")
            print("4. Display data types")
            print("5. Display basic information")
            print("6. Display dataset shape")
            print("7. Convert selected columns to NumPy arrays and demonstrate indexing/slicing")
            print("8. Back to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "1":
                print(self.data.head())
            elif choice == "2":
                print(self.data.tail())
            elif choice == "3":
                print(list(self.data.columns))
            elif choice == "4":
                print(self.data.dtypes)
            elif choice == "5":
                self.data.info()
            elif choice == "6":
                print(f"Rows: {self.data.shape[0]}, Columns: {self.data.shape[1]}")
            elif choice == "7":
                self.numpy_array_operations()
            elif choice == "8":
                break
            else:
                print("Invalid choice. Please enter 1-8.")

    def numpy_array_operations(self):
        if not self._ready():
            return
        numeric_cols = self.data.select_dtypes(include=np.number).columns.tolist()
        if not numeric_cols:
            print("No numeric columns are available for NumPy conversion.")
            return
        print("Numeric columns:", ", ".join(numeric_cols))
        col = input("Enter a numeric column name: ").strip()
        if col not in numeric_cols:
            print("Column not found or it is not numeric.")
            return
        arr = self.data[col].to_numpy()
        print("\nNumPy array:")
        print(arr)
        print("Array shape:", arr.shape)
        print("Array data type:", arr.dtype)
        print("First element (index 0):", arr[0] if len(arr) else "Array is empty")
        print("First five elements (slice [:5]):", arr[:5])
        start = input("Enter slice start index (default 0): ").strip()
        end = input("Enter slice end index (default 5): ").strip()
        try:
            start_idx = int(start) if start else 0
            end_idx = int(end) if end else 5
            print("Selected slice:", arr[start_idx:end_idx])
        except ValueError:
            print("Invalid index; showing default slice:", arr[:5])

    def clean_data(self):
        if not self._ready():
            return
        while True:
            print("\n== Handle Missing Data / Clean Data ==")
            print("1. Display rows containing missing values")
            print("2. Fill numeric missing values with column mean")
            print("3. Drop rows containing missing values")
            print("4. Replace missing values with a specific value")
            print("5. Remove duplicate rows")
            print("6. Convert a column to numeric")
            print("7. Back to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "1":
                missing = self.data[self.data.isna().any(axis=1)]
                if missing.empty:
                    print("No missing values found in the dataset!")
                else:
                    print(missing)
                    print("\nMissing values per column:\n", self.data.isna().sum())
            elif choice == "2":
                numeric = self.data.select_dtypes(include=np.number).columns
                if len(numeric) == 0:
                    print("No numeric columns available.")
                else:
                    self.data[numeric] = self.data[numeric].fillna(self.data[numeric].mean())
                    print("Missing numeric values filled with their column means.")
            elif choice == "3":
                before = len(self.data)
                self.data.dropna(inplace=True)
                self.data.reset_index(drop=True, inplace=True)
                print(f"Dropped {before - len(self.data)} row(s).")
            elif choice == "4":
                value = input("Enter replacement value (text or number): ")
                try:
                    replacement = float(value)
                    if replacement.is_integer():
                        replacement = int(replacement)
                except ValueError:
                    replacement = value
                self.data.fillna(replacement, inplace=True)
                print("Missing values replaced.")
            elif choice == "5":
                before = len(self.data)
                self.data.drop_duplicates(inplace=True)
                self.data.reset_index(drop=True, inplace=True)
                print(f"Removed {before - len(self.data)} duplicate row(s).")
            elif choice == "6":
                col = input("Enter column name to convert to numeric: ").strip()
                if col not in self.data.columns:
                    print("Column not found.")
                else:
                    self.data[col] = pd.to_numeric(self.data[col], errors="coerce")
                    print(f"Column '{col}' converted; unconvertible values became missing values.")
            elif choice == "7":
                break
            else:
                print("Invalid choice. Please enter 1-7.")

    def mathematical_operations(self):
        if not self._ready():
            return
        numeric = self.data.select_dtypes(include=np.number).columns.tolist()
        if not numeric:
            print("No numeric columns available.")
            return
        print("Numeric columns:", ", ".join(numeric))
        col = input("Choose a numeric column: ").strip()
        if col not in numeric:
            print("Column not found or not numeric.")
            return
        series = self.data[col]
        print(f"\nOperations for '{col}':")
        print("1. Add a value")
        print("2. Multiply by a value")
        print("3. Subtract a value")
        print("4. Divide by a value")
        print("5. Show square and square root")
        choice = input("Enter your choice: ").strip()
        if choice == "5":
            print(pd.DataFrame({"original": series, "square": series ** 2,
                                "square_root": np.sqrt(series.where(series >= 0))}).head(20))
            return
        if choice not in {"1", "2", "3", "4"}:
            print("Invalid choice.")
            return
        try:
            value = float(input("Enter a numeric value: "))
            if choice == "1":
                result = series + value
            elif choice == "2":
                result = series * value
            elif choice == "3":
                result = series - value
            else:
                if value == 0:
                    print("Cannot divide by zero.")
                    return
                result = series / value
            print(pd.DataFrame({"original": series, "result": result}).head(20))
        except ValueError:
            print("Please enter a valid number.")

    def combine_data(self):
        if not self._ready():
            return
        path = input("Enter path to the second CSV file: ").strip().strip('"')
        if not os.path.isfile(path):
            print("File not found.")
            return
        try:
            other = pd.read_csv(path)
            print("\n1. Concatenate rows")
            print("2. Merge datasets using a common column")
            print("3. Join datasets by index")
            choice = input("Choose operation: ").strip()
            if choice == "1":
                combined = pd.concat([self.data, other], ignore_index=True, sort=False)
            elif choice == "2":
                common = sorted(set(self.data.columns).intersection(other.columns))
                if not common:
                    print("No common columns found to merge on.")
                    return
                print("Common columns:", ", ".join(common))
                key = input("Enter merge column: ").strip()
                if key not in common:
                    print("Invalid merge column.")
                    return
                combined = pd.merge(self.data, other, on=key, how="outer", suffixes=("_left", "_right"))
            elif choice == "3":
                combined = self.data.join(other, lsuffix="_left", rsuffix="_right", how="outer")
            else:
                print("Invalid choice.")
                return
            print(combined.head(20))
            save = input("Replace current dataset with combined data? (y/n): ").strip().lower()
            if save == "y":
                self.data = combined
                print("Current dataset updated in memory.")
        except Exception as exc:
            print(f"Could not combine datasets: {exc}")

    def split_data(self):
        if not self._ready():
            return
        col = input("Enter column to split data by: ").strip()
        if col not in self.data.columns:
            print("Column not found.")
            return
        value = input("Enter the value for the first subset: ").strip()
        # Compare as strings so CSV values entered by users are easy to match.
        mask = self.data[col].astype(str).str.casefold() == value.casefold()
        first = self.data[mask]
        second = self.data[~mask]
        print(f"\nSubset where {col} = {value!r} ({len(first)} rows):")
        print(first.head(20))
        print(f"\nRemaining subset ({len(second)} rows):")
        print(second.head(20))
        save = input("Save the two subsets as separate CSV files? (y/n): ").strip().lower()
        if save == "y":
            first.to_csv("split_matching_rows.csv", index=False)
            second.to_csv("split_remaining_rows.csv", index=False)
            print("Saved split_matching_rows.csv and split_remaining_rows.csv.")

    def search_sort_filter(self):
        if not self._ready():
            return
        while True:
            print("\n== Search, Sort and Filter ==")
            print("1. Search for a value")
            print("2. Sort by a column")
            print("3. Filter numeric values using a threshold")
            print("4. Filter by text/category")
            print("5. Back to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "1":
                col = input("Enter column name: ").strip()
                if col not in self.data.columns:
                    print("Column not found.")
                    continue
                value = input("Enter search text/value: ").strip()
                result = self.data[self.data[col].astype(str).str.contains(value, case=False, na=False)]
                print(result if not result.empty else "No matching records found.")
            elif choice == "2":
                col = input("Enter column name to sort by: ").strip()
                if col not in self.data.columns:
                    print("Column not found.")
                    continue
                order = input("Ascending order? (y/n): ").strip().lower() != "n"
                print(self.data.sort_values(by=col, ascending=order, na_position="last").head(50))
            elif choice == "3":
                numeric = self.data.select_dtypes(include=np.number).columns.tolist()
                print("Numeric columns:", ", ".join(numeric) if numeric else "None")
                col = input("Enter numeric column: ").strip()
                if col not in numeric:
                    print("Column not found or not numeric.")
                    continue
                try:
                    threshold = float(input("Enter threshold: "))
                    print(self.data[self.data[col] >= threshold])
                except ValueError:
                    print("Please enter a valid number.")
            elif choice == "4":
                col = input("Enter column name: ").strip()
                if col not in self.data.columns:
                    print("Column not found.")
                    continue
                value = input("Enter exact category/value: ").strip()
                print(self.data[self.data[col].astype(str).str.casefold() == value.casefold()])
            elif choice == "5":
                break
            else:
                print("Invalid choice. Please enter 1-5.")

    def aggregate_functions(self):
        if not self._ready():
            return
        numeric = self.data.select_dtypes(include=np.number).columns.tolist()
        if not numeric:
            print("No numeric columns available.")
            return
        print("Numeric columns:", ", ".join(numeric))
        col = input("Enter numeric column to aggregate: ").strip()
        if col not in numeric:
            print("Column not found or not numeric.")
            return
        series = self.data[col]
        print("\nAggregation results:")
        print("Sum:", series.sum())
        print("Mean:", series.mean())
        print("Median:", series.median())
        print("Minimum:", series.min())
        print("Maximum:", series.max())
        print("Count:", series.count())
        print("Standard deviation:", series.std())
        group_col = input("Optional: enter a category column for group-wise totals (press Enter to skip): ").strip()
        if group_col:
            if group_col not in self.data.columns:
                print("Column not found.")
            else:
                print(self.data.groupby(group_col)[col].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False))

    def statistical_analysis(self):
        if not self._ready():
            return
        print("\n== Descriptive Statistics ==")
        print(self.data.describe(include="all").transpose())
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns available for standard deviation, variance and percentiles.")
            return
        print("\nStandard deviation:\n", numeric.std())
        print("\nVariance:\n", numeric.var())
        print("\n25th, 50th and 75th percentiles:\n", numeric.quantile([0.25, 0.50, 0.75]))
        print("\nCorrelation matrix:\n", numeric.corr(numeric_only=True))

    def create_pivot_table(self):
        if not self._ready():
            return
        print("Available columns:", ", ".join(map(str, self.data.columns)))
        index_col = input("Enter grouping/index column: ").strip()
        if index_col not in self.data.columns:
            print("Index column not found.")
            return
        values = input("Enter numeric values column: ").strip()
        numeric = self.data.select_dtypes(include=np.number).columns.tolist()
        if values not in numeric:
            print("Values column must be numeric. Numeric columns:", ", ".join(numeric))
            return
        agg = input("Aggregation (sum/mean/count/max/min; default sum): ").strip().lower() or "sum"
        if agg not in {"sum", "mean", "count", "max", "min"}:
            print("Unsupported aggregation; using sum.")
            agg = "sum"
        pivot = pd.pivot_table(self.data, index=index_col, values=values, aggfunc=agg, fill_value=0)
        print("\nPivot table:\n", pivot)
        save = input("Save pivot table as pivot_table.csv? (y/n): ").strip().lower()
        if save == "y":
            pivot.to_csv("pivot_table.csv")
            print("Saved pivot_table.csv.")

    def visualize_data(self):
        if not self._ready():
            return
        while True:
            print("\n== Data Visualization ==")
            print("1. Bar Plot")
            print("2. Line Plot")
            print("3. Scatter Plot")
            print("4. Pie Chart")
            print("5. Histogram")
            print("6. Stack Plot")
            print("7. Seaborn Heatmap")
            print("8. Seaborn Box Plot")
            print("9. Seaborn Count Plot")
            print("10. Display multiple plots in one figure (subplots)")
            print("11. Back to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "11":
                break
            if choice not in {str(i) for i in range(1, 11)}:
                print("Invalid choice. Please enter 1-11.")
                continue
            try:
                if choice == "7":
                    numeric = self.data.select_dtypes(include=np.number)
                    if numeric.empty:
                        print("Heatmap needs numeric columns.")
                        continue
                    plt.figure(figsize=(9, 6))
                    sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm", fmt=".2f")
                    plt.title("Correlation Heatmap")
                elif choice == "8":
                    col = self._choose_column("Choose numeric column for box plot: ", numeric_only=True)
                    if col is None:
                        continue
                    plt.figure(figsize=(8, 5))
                    sns.boxplot(y=self.data[col])
                    plt.title(f"Box Plot of {col}")
                elif choice == "9":
                    col = self._choose_column("Choose category column for count plot: ")
                    if col is None:
                        continue
                    plt.figure(figsize=(9, 5))
                    sns.countplot(data=self.data, x=col, order=self.data[col].astype(str).value_counts().index[:15])
                    plt.title(f"Count Plot of {col}")
                    plt.xticks(rotation=45, ha="right")
                elif choice == "10":
                    self._multiple_plots()
                    continue
                else:
                    self._basic_plot(choice)
                    continue
                plt.tight_layout()
                plt.show()
                self.last_plot = plt.gcf()
                print("Visualization displayed successfully.")
            except (ValueError, TypeError, KeyError) as exc:
                plt.close("all")
                print(f"Could not create this plot: {exc}")

    def _choose_column(self, prompt, numeric_only=False):
        columns = (self.data.select_dtypes(include=np.number).columns.tolist()
                   if numeric_only else self.data.columns.tolist())
        print("Available columns:", ", ".join(map(str, columns)) if columns else "None")
        col = input(prompt).strip()
        if col not in columns:
            print("Column not found or unsuitable for this plot.")
            return None
        return col

    def _basic_plot(self, choice):
        if choice == "1":
            x = self._choose_column("Enter category/X-axis column: ")
            y = self._choose_column("Enter numeric/Y-axis column: ", True)
            if x is None or y is None:
                return
            grouped = self.data.groupby(x, dropna=False)[y].sum().sort_values(ascending=False).head(20)
            plt.figure(figsize=(9, 5))
            grouped.plot(kind="bar")
            plt.xlabel(x)
            plt.ylabel(f"Sum of {y}")
            plt.title(f"{y} by {x}")
            plt.xticks(rotation=45, ha="right")
        elif choice == "2":
            x = self._choose_column("Enter X-axis column (often Date/Year): ")
            y = self._choose_column("Enter numeric Y-axis column: ", True)
            if x is None or y is None:
                return
            temp = self.data[[x, y]].copy()
            temp = temp.sort_values(x)
            plt.figure(figsize=(9, 5))
            plt.plot(temp[x].astype(str), temp[y], marker="o")
            plt.xlabel(x)
            plt.ylabel(y)
            plt.title(f"{y} Trend")
            plt.xticks(rotation=45, ha="right")
        elif choice == "3":
            x = self._choose_column("Enter numeric X-axis column: ", True)
            y = self._choose_column("Enter numeric Y-axis column: ", True)
            if x is None or y is None:
                return
            plt.figure(figsize=(8, 5))
            plt.scatter(self.data[x], self.data[y], alpha=0.75)
            plt.xlabel(x)
            plt.ylabel(y)
            plt.title(f"Scatter Plot: {x} vs {y}")
        elif choice == "4":
            col = self._choose_column("Enter category column for pie chart: ")
            if col is None:
                return
            counts = self.data[col].astype(str).value_counts().head(10)
            plt.figure(figsize=(7, 7))
            plt.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90)
            plt.title(f"Distribution of {col}")
        elif choice == "5":
            col = self._choose_column("Enter numeric column for histogram: ", True)
            if col is None:
                return
            plt.figure(figsize=(8, 5))
            plt.hist(self.data[col].dropna(), bins=10, edgecolor="black")
            plt.xlabel(col)
            plt.ylabel("Frequency")
            plt.title(f"Histogram of {col}")
        elif choice == "6":
            numeric = self.data.select_dtypes(include=np.number).columns.tolist()
            if len(numeric) < 2:
                print("Stack plot requires at least two numeric columns.")
                return
            print("Numeric columns:", ", ".join(numeric))
            chosen = input("Enter 2 or more numeric column names separated by commas: ").split(",")
            chosen = [c.strip() for c in chosen if c.strip() in numeric]
            if len(chosen) < 2:
                print("Please select at least two valid numeric columns.")
                return
            values = self.data[chosen].fillna(0).head(30)
            plt.figure(figsize=(9, 5))
            plt.stackplot(range(len(values)), *[values[c].to_numpy() for c in chosen], labels=chosen)
            plt.xlabel("Row")
            plt.ylabel("Value")
            plt.title("Stack Plot of Numeric Columns")
            plt.legend(loc="upper left")
        plt.tight_layout()
        plt.show()
        self.last_plot = plt.gcf()
        print("Visualization displayed successfully.")

    def _multiple_plots(self):
        numeric = self.data.select_dtypes(include=np.number).columns.tolist()
        if not numeric:
            print("No numeric columns available.")
            return
        col = input("Enter numeric column for subplot examples: ").strip()
        if col not in numeric:
            print("Column not found or not numeric.")
            return
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].hist(self.data[col].dropna(), bins=10, edgecolor="black")
        axes[0].set_title("Histogram")
        axes[1].boxplot(self.data[col].dropna())
        axes[1].set_title("Box Plot")
        axes[2].plot(self.data[col].reset_index(drop=True))
        axes[2].set_title("Line Plot")
        fig.suptitle(f"Multiple Views of {col}")
        fig.tight_layout()
        plt.show()
        self.last_plot = fig
        print("Multiple plots displayed successfully.")

    def save_visualization(self):
        if not self._ready():
            return
        print("Create a plot using main menu option 6 before saving.")
        filename = input("Enter image filename (e.g., sales_plot.png): ").strip()
        if not filename:
            print("Filename cannot be empty.")
            return
        if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".pdf", ".svg")):
            filename += ".png"
        try:
            # Saving the currently active Matplotlib figure is supported.
            fig = plt.gcf()
            if not fig.axes:
                print("No active plot found. Please create a plot first.")
                return
            fig.savefig(filename, bbox_inches="tight", dpi=300)
            print(f"Visualization saved as '{filename}' successfully!")
        except (OSError, ValueError) as exc:
            print(f"Could not save visualization: {exc}")

In [4]:
# Notebook-friendly main menu
analyzer = SalesDataAnalyzer()

main_options = [
    ("1. Load Dataset", "load"),
    ("2. Explore Data", "explore"),
    ("3. DataFrame / Mathematical Operations", "operations"),
    ("4. Handle Missing Data and Clean Dataset", "clean"),
    ("5. Generate Descriptive Statistics", "statistics"),
    ("6. Data Visualization", "visualize"),
    ("7. Save Visualization", "save_plot"),
    ("8. Combine Datasets", "combine"),
    ("9. Split Dataset", "split"),
    ("10. Search, Sort and Filter Data", "search"),
    ("11. Aggregation Functions", "aggregate"),
    ("12. Create Pivot Table", "pivot"),
    ("13. Exit", "exit"),
]

menu_dropdown = widgets.Dropdown(
    options=main_options,
    value="load",
    description="Menu:",
    layout=widgets.Layout(width="520px")
)
csv_path = widgets.Text(
    value="sample_sales_data.csv",
    description="CSV path:",
    placeholder="e.g. C:/Users/YourName/Desktop/sales_data.csv",
    layout=widgets.Layout(width="720px")
)
operation_dropdown = widgets.Dropdown(
    options=[
        ("Mathematical operations", "math"),
        ("Combine DataFrames", "combine"),
        ("Split DataFrame", "split"),
    ],
    value="math",
    description="Option:",
    layout=widgets.Layout(width="420px")
)
run_button = widgets.Button(description="Run Selected Option", button_style="primary", icon="play")
responses_box = widgets.Textarea(
    value="",
    description="Answers:",
    placeholder="Enter answers in prompt order, one per line. Example: 1 then 7",
    layout=widgets.Layout(width="720px", height="110px")
)
menu_output = widgets.Output()

def run_notebook_option(_=None):
    global _notebook_input_answers
    selected = menu_dropdown.value
    _notebook_input_answers = responses_box.value.splitlines()

    # Clear the previous result before running the newly selected option.
    menu_output.clear_output(wait=True)
    with menu_output:
        try:
            _run_selected_option(selected)
        except NotebookInputCancelled as exc:
            print(exc)
        except EOFError:
            print("This option needs more answers. Enter all requested answers in the Answers box, one per line, then click Run Selected Option again.")
        except Exception as exc:
            print(f"Could not complete this option: {exc}")

def _run_selected_option(selected):
        if selected == "load":
            analyzer.load_data(csv_path.value.strip().strip('"'))
        elif selected == "explore":
            analyzer.explore_data()
        elif selected == "operations":
            sub = operation_dropdown.value
            if sub == "math":
                analyzer.mathematical_operations()
            elif sub == "combine":
                analyzer.combine_data()
            elif sub == "split":
                analyzer.split_data()
        elif selected == "clean":
            analyzer.clean_data()
        elif selected == "statistics":
            analyzer.statistical_analysis()
        elif selected == "visualize":
            analyzer.visualize_data()
        elif selected == "save_plot":
            analyzer.save_visualization()
        elif selected == "combine":
            analyzer.combine_data()
        elif selected == "split":
            analyzer.split_data()
        elif selected == "search":
            analyzer.search_sort_filter()
        elif selected == "aggregate":
            analyzer.aggregate_functions()
        elif selected == "pivot":
            analyzer.create_pivot_table()
        elif selected == "exit":
            print("Thank you for using Pandas Analyzer & Data Visualization.")

run_button.on_click(run_notebook_option)

display(widgets.HTML("<h3>Data Analysis & Visualization Menu</h3>"))
display(menu_dropdown)
display(csv_path)
display(widgets.HTML("<b>For options that ask questions, enter answers below in prompt order, one per line.</b>"))
display(responses_box)
display(widgets.HTML("<b>For menu option 3, choose a sub-operation here:</b>"))
display(operation_dropdown)
display(run_button)
display(menu_output)

print("Ready. Load a CSV file first, or use the optional sample_sales_data.csv cell.")


HTML(value='<h3>Data Analysis & Visualization Menu</h3>')

Dropdown(description='Menu:', layout=Layout(width='520px'), options=(('1. Load Dataset', 'load'), ('2. Explore…

Text(value='sample_sales_data.csv', description='CSV path:', layout=Layout(width='720px'), placeholder='e.g. C…

HTML(value='<b>For options that ask questions, enter answers below in prompt order, one per line.</b>')

Textarea(value='', description='Answers:', layout=Layout(height='110px', width='720px'), placeholder='Enter an…

HTML(value='<b>For menu option 3, choose a sub-operation here:</b>')

Dropdown(description='Option:', layout=Layout(width='420px'), options=(('Mathematical operations', 'math'), ('…

Button(button_style='primary', description='Run Selected Option', icon='play', style=ButtonStyle())

Output()

Ready. Load a CSV file first, or use the optional sample_sales_data.csv cell.
